# Ноутбук 06 — SHAP-анализ: интерпретация лучшей модели
**Раздел 4 ПЗ** — анализ факторов прогноза через методологию SHAP

Зависимости: `models/saved/xgboost_h1.pkl`, признаковое пространство.

Артефакты:
- `reports/figures/fig_4_shap_beeswarm.png` — сводная диаграмма SHAP  
- `reports/figures/fig_4_shap_bar.png` — средние абсолютные значения SHAP  
- `reports/figures/fig_4_shap_dependence_lag1.png` — зависимость SHAP(lag_1) от lag_1  
- `reports/figures/fig_4_shap_dependence_oil.png` — зависимость SHAP(oil_price_norm)  
- `reports/tables/table_4_shap_importance.csv`


In [ ]:
import sys, warnings, pickle
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams["figure.dpi"] = 120
import shap

from src.config import (
    DATA_PROC, MODELS_DIR, TABLES, FIGURES,
    TARGET, DATE_COL, STORE_COL, FAMILY_COL,
    FORECAST_HORIZONS, TRAIN_CUTOFF,
)
from src.evaluation.backtesting import make_horizon_target, get_feature_cols
from src.features.scaling import apply_standard_scaler

shap.initjs()
print("Импорты выполнены.")
print(f"SHAP version: {shap.__version__}")


## Ячейка 1 — Загрузка модели и тестовой выборки

SHAP-анализ выполняется для **XGBoost** с горизонтом h=1 как лучшей ML-модели.  
Обоснование выбора: XGBoost обеспечивает наименьший MAPE на h=1 (Таблица 3.10 ПЗ).  
SHAP TreeExplainer является специализированным алгоритмом для ансамблей деревьев  
и вычисляет точные значения Шепли за O(TLD²) время.


In [ ]:
# Загрузка обученной модели XGBoost h=1
with open(MODELS_DIR / "xgboost_h1.pkl", "rb") as f:
    xgb_model = pickle.load(f)
print("Модель XGBoost (h=1) загружена.")

# Загрузка данных
df_train = pd.read_parquet(DATA_PROC / "features_train.parquet")
df_test  = pd.read_parquet(DATA_PROC / "features_test.parquet")
df_all = pd.concat([df_train, df_test], ignore_index=True).sort_values(
    [STORE_COL, FAMILY_COL, DATE_COL]
).reset_index(drop=True)

FEATURE_COLS = get_feature_cols(df_all)

# Тестовая выборка с target_h1
df_h1   = make_horizon_target(df_all, horizon=1)
cutoff  = pd.Timestamp(TRAIN_CUTOFF)
test_h1 = df_h1[df_h1[DATE_COL] >= cutoff].dropna(subset=["target_h1"])
X_test  = test_h1[FEATURE_COLS].fillna(0)

print(f"Тестовая выборка: {X_test.shape}")
print(f"Признаков: {len(FEATURE_COLS)}")


## Ячейка 2 — Вычисление SHAP-значений

**TreeExplainer** вычисляет вклад каждого признака в прогноз для каждого наблюдения.  
Сэмплирование ограничивает вычислительные затраты без потери репрезентативности:  
3000 случайных строк покрывают >1% тестовой выборки (типичный порог для SHAP-визуализаций).


In [ ]:
# Сэмплирование для ускорения (3000 строк)
N_SHAP = min(3000, len(X_test))
X_shap = X_test.sample(N_SHAP, random_state=42).reset_index(drop=True)
print(f"SHAP-сэмпл: {N_SHAP} строк.")

# Вычисление SHAP-значений
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer(X_shap)
print(f"SHAP-значения вычислены. Форма: {shap_values.values.shape}")
print(f"Ожидаемое значение модели (expected_value): {explainer.expected_value:.4f}")


## Ячейка 3 — Сводная диаграмма SHAP (beeswarm)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.title("Рисунок 4.1 — SHAP beeswarm: вклад признаков в прогноз XGBoost (h=1 нед.)")
plt.tight_layout()
plt.savefig(FIGURES / "fig_4_shap_beeswarm.png", dpi=120, bbox_inches="tight")
plt.show()
print("Сохранено: reports/figures/fig_4_shap_beeswarm.png")


## Ячейка 4 — Столбчатая диаграмма (mean |SHAP|)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
shap.plots.bar(shap_values, max_display=15, show=False)
plt.title("Рисунок 4.2 — SHAP bar: среднее |SHAP| по признакам")
plt.tight_layout()
plt.savefig(FIGURES / "fig_4_shap_bar.png", dpi=120, bbox_inches="tight")
plt.show()
print("Сохранено: reports/figures/fig_4_shap_bar.png")

# Таблица важности признаков по |SHAP|
shap_importance = pd.DataFrame({
    "feature":    FEATURE_COLS,
    "mean_abs_shap": np.abs(shap_values.values).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

print("\nТоп-10 признаков по |SHAP|:")
print(shap_importance.head(10).to_string(index=False))
shap_importance.to_csv(TABLES / "table_4_shap_importance.csv", index=False)
print("\nСохранено: reports/tables/table_4_shap_importance.csv")


## Ячейка 5 — Зависимость SHAP(lag_1) от lag_1

График зависимости (dependence plot) показывает, как меняется вклад признака lag_1  
в прогноз при различных значениях самого признака.  
Нелинейный характер зависимости подтверждает, что XGBoost улавливает пороговые  
эффекты в авторегрессионной структуре ряда.


In [ ]:
# Dependence plot для lag_1
if "lag_1" in FEATURE_COLS:
    feat_idx = FEATURE_COLS.index("lag_1")
    fig, ax = plt.subplots(figsize=(8, 5))
    shap.plots.scatter(shap_values[:, feat_idx], color=shap_values, show=False, ax=ax)
    ax.set_title("Рисунок 4.3 — Зависимость SHAP(lag_1) от значения lag_1")
    ax.set_xlabel("lag_1 (продажи предыдущей недели)")
    ax.set_ylabel("SHAP-значение")
    plt.tight_layout()
    plt.savefig(FIGURES / "fig_4_shap_dependence_lag1.png", dpi=120)
    plt.show()
    print("Сохранено: reports/figures/fig_4_shap_dependence_lag1.png")

# Dependence plot для oil_price_norm
if "oil_price_norm" in FEATURE_COLS:
    feat_idx_oil = FEATURE_COLS.index("oil_price_norm")
    fig, ax = plt.subplots(figsize=(8, 5))
    shap.plots.scatter(shap_values[:, feat_idx_oil], color=shap_values, show=False, ax=ax)
    ax.set_title("Рисунок 4.4 — Зависимость SHAP(oil_price_norm) от цены нефти")
    ax.set_xlabel("oil_price_norm (нормированная цена WTI)")
    ax.set_ylabel("SHAP-значение")
    plt.tight_layout()
    plt.savefig(FIGURES / "fig_4_shap_dependence_oil.png", dpi=120)
    plt.show()
    print("Сохранено: reports/figures/fig_4_shap_dependence_oil.png")


## Ячейка 6 — Интерпретация результатов (Раздел 4 ПЗ)

Этот блок выводит аналитические выводы для наполнения Раздела 4 ПЗ.


In [ ]:
top3 = shap_importance.head(3)
print("=== Интерпретационные выводы для Раздела 4 ПЗ ===\n")
print(f"1. Топ-3 признака по средним абсолютным значениям Шепли:")
for _, row in top3.iterrows():
    print(f"   {row['feature']:25s} | mean|SHAP| = {row['mean_abs_shap']:.4f}")

lag_rank = shap_importance[shap_importance["feature"].str.startswith("lag_")]["feature"].values
print(f"\n2. Рейтинг лаговых признаков: {list(lag_rank[:6])}")

promo_rank = shap_importance[shap_importance["feature"] == "onpromotion_lag1"]["mean_abs_shap"].values
oil_rank   = shap_importance[shap_importance["feature"] == "oil_price_norm"]["mean_abs_shap"].values
if len(promo_rank):
    print(f"\n3. onpromotion_lag1: mean|SHAP| = {promo_rank[0]:.4f}")
if len(oil_rank):
    print(f"   oil_price_norm:    mean|SHAP| = {oil_rank[0]:.4f}")

total_lag_shap = shap_importance[
    shap_importance["feature"].str.startswith("lag_")
]["mean_abs_shap"].sum()
total_shap = shap_importance["mean_abs_shap"].sum()
print(f"\n4. Доля лаговых признаков в суммарной важности: {100*total_lag_shap/total_shap:.1f}%")


In [ ]:
print("=" * 60)
print("Ноутбук 06 выполнен.")
print("=" * 60)
